[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/23_probabilistic_machine_learning/04_gaussian_processes.ipynb)

# Bài 04 — Gaussian Process

**🎯 Mục tiêu**
- Từ hồi quy Bayes trên trọng số sang **phân phối trên hàm**: kernel = hiệp phương sai giữa giá trị hàm.
- Lấy mẫu từ GP prior với các kernel khác nhau (RBF, periodic, linear, tổng/tích).
- **GP regression from scratch** bằng Cholesky (Rasmussen & Williams, Alg. 2.1) — chính là Gaussian có điều kiện (bài 01).
- Chọn siêu tham số bằng **log marginal likelihood**; `GaussianProcessRegressor`; ngoại suy với kernel phù hợp.
- Chi phí $O(n^3)$ và các hướng xấp xỉ; GP trong Bayesian optimization.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/23_probabilistic_machine_learning

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, DotProduct, ExpSineSquared, WhiteKernel

from ml_course.visualization import set_style

set_style()
warnings.filterwarnings("ignore", category=ConvergenceWarning)
rng = np.random.default_rng(0)

## 1. Từ trọng số đến hàm

Hồi quy Bayes (bài 03): $f(x) = \mathbf w^\top\boldsymbol\phi(x)$, $\mathbf w \sim \mathcal N(\mathbf 0, \Sigma_p)$. Khi đó với bất kỳ tập điểm $x_1, \dots, x_n$, vector $[f(x_1), \dots, f(x_n)]$ là Gaussian với
$$\text{Cov}(f(x), f(x')) = \boldsymbol\phi(x)^\top\Sigma_p\boldsymbol\phi(x') =: k(x, x')$$

Ý tưởng GP: **bỏ qua trọng số**, chỉ định thẳng **kernel** $k$. Định nghĩa: $f \sim \mathcal{GP}(m, k)$ nếu mọi tập hữu hạn giá trị hàm có phân phối Gaussian chung với trung bình $m(x_i)$ và hiệp phương sai $K_{ij} = k(x_i, x_j)$. Kernel RBF tương ứng với **vô số** hàm cơ sở Gaussian — điều không thể làm trực tiếp trong không gian trọng số.

**RBF**: $k(x, x') = \sigma_f^2\exp\left(-\frac{(x - x')^2}{2\ell^2}\right)$ — hai điểm gần nhau (so với **length scale** $\ell$) có giá trị hàm tương quan mạnh → hàm trơn.

## 2. Lấy mẫu từ GP prior

In [ ]:
def rbf(a, b, ell=1.0, sf=1.0):
    return sf**2 * np.exp(-0.5 * (a[:, None] - b[None]) ** 2 / ell**2)


def periodic(a, b, ell=1.0, period=1.0):
    return np.exp(-2 * np.sin(np.pi * np.abs(a[:, None] - b[None]) / period) ** 2 / ell**2)


def linear(a, b):
    return a[:, None] * b[None]


xs = np.linspace(-5, 5, 200)
kernels = {
    "RBF ℓ = 0.3": rbf(xs, xs, 0.3), "RBF ℓ = 1": rbf(xs, xs, 1.0), "RBF ℓ = 3": rbf(xs, xs, 3.0),
    "Periodic (chu kỳ 2.5)": periodic(xs, xs, 1.0, 2.5), "Linear": 0.1 * linear(xs, xs),
    "Linear + Periodic": 0.05 * linear(xs, xs) + periodic(xs, xs, 1.0, 2.5),
}
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, (name, K) in zip(axes.ravel(), kernels.items()):
    Lk = np.linalg.cholesky(K + 1e-6 * np.eye(len(xs)))                 # jitter cho ổn định số
    ax.plot(xs, Lk @ rng.standard_normal((len(xs), 4)))
    ax.set_title(name)
plt.tight_layout()
plt.show()

Kernel mã hoá **giả định** về hàm: trơn đến mức nào (ℓ), có tuần hoàn, có xu hướng tuyến tính... **Tổng** kernel = cộng các thành phần (xu hướng + mùa vụ); **tích** kernel = điều biến (mùa vụ có biên độ thay đổi).

## 3. GP regression from scratch

Quan sát $\mathbf y = f(X) + \boldsymbol\varepsilon$, $\varepsilon \sim \mathcal N(0, \sigma_n^2)$. Phân phối chung của $\mathbf y$ và $\mathbf f_* = f(X_*)$:
$$\begin{bmatrix}\mathbf y \\ \mathbf f_*\end{bmatrix} \sim \mathcal N\left(\mathbf 0, \begin{bmatrix}K + \sigma_n^2 I & K_* \\ K_*^\top & K_{**}\end{bmatrix}\right)$$
Áp dụng **Gaussian có điều kiện** (bài 01):
$$\bar{\mathbf f}_* = K_*^\top(K + \sigma_n^2I)^{-1}\mathbf y, \qquad \text{Cov}(\mathbf f_*) = K_{**} - K_*^\top(K + \sigma_n^2I)^{-1}K_*$$

Cài đặt ổn định số bằng Cholesky, không nghịch đảo ma trận:

In [ ]:
def gp_predict(X, y, X_star, kernel, noise):
    K = kernel(X, X) + noise**2 * np.eye(len(X))
    L = np.linalg.cholesky(K)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))                 # (K + σ²I)⁻¹ y
    K_s = kernel(X, X_star)
    mean = K_s.T @ alpha
    v = np.linalg.solve(L, K_s)
    var = np.diag(kernel(X_star, X_star)) - np.sum(v**2, axis=0)
    log_ml = -0.5 * y @ alpha - np.sum(np.log(np.diag(L))) - len(X) / 2 * np.log(2 * np.pi)
    return mean, var, log_ml


f_true = lambda x: np.sin(x) + 0.3 * x
X_tr = np.r_[rng.uniform(-4, -1, 8), rng.uniform(1.5, 4, 8)]
y_tr = f_true(X_tr) + rng.normal(0, 0.2, len(X_tr))
xg = np.linspace(-6, 6, 300)
mean, var, lml = gp_predict(X_tr, y_tr, xg, lambda a, b: rbf(a, b, 1.0, 1.0), noise=0.2)
sd = np.sqrt(np.maximum(var, 0))
plt.figure(figsize=(11, 4.5))
plt.plot(xg, f_true(xg), "g--", label="hàm thật")
plt.plot(xg, mean, "r", label="trung bình hậu nghiệm")
plt.fill_between(xg, mean - 2 * sd, mean + 2 * sd, color="r", alpha=0.2, label="±2σ")
plt.scatter(X_tr, y_tr, color="k", zorder=3)
plt.legend()
plt.title(f"GP với RBF ℓ = 1 — log marginal likelihood = {lml:.2f}")
plt.show()

ref = GaussianProcessRegressor(ConstantKernel(1.0, "fixed") * RBF(1.0, "fixed"), alpha=0.2**2, optimizer=None).fit(X_tr[:, None], y_tr)
m_ref, s_ref = ref.predict(xg[:, None], return_std=True)
print(f"so với sklearn: |Δmean| max = {np.abs(mean - m_ref).max():.1e} | |Δstd| max = {np.abs(sd - s_ref).max():.1e} | "
      f"Δ log-ML = {abs(lml - ref.log_marginal_likelihood_value_):.1e}")

Giống hồi quy Bayes: độ bất định nhỏ gần dữ liệu, lớn trong khoảng trống; **ngoài vùng dữ liệu**, RBF quên dần dữ liệu → trung bình về 0 (trung bình prior) và độ bất định về mức prior $\sigma_f$ — GP "thành thật" hơn hàm cơ sở cố định của bài 03, nhưng **không ngoại suy được xu hướng** $0.3x$.

## 4. Siêu tham số và log marginal likelihood

$$\log p(\mathbf y \mid X, \theta) = -\tfrac12\mathbf y^\top(K + \sigma_n^2I)^{-1}\mathbf y - \tfrac12\log|K + \sigma_n^2I| - \tfrac n2\log 2\pi$$
Số hạng đầu: **khớp dữ liệu**; số hạng thứ hai: **phạt độ phức tạp** — cùng tinh thần evidence ở bài 03.

In [ ]:
ells = [0.1, 0.3, 1.0, 3.0, 10.0]
fig, axes = plt.subplots(1, len(ells), figsize=(25, 4))
lml_by_ell = {}
for ax, ell in zip(axes, ells):
    m, v, lml_e = gp_predict(X_tr, y_tr, xg, lambda a, b, e=ell: rbf(a, b, e, 1.0), noise=0.2)
    lml_by_ell[ell] = lml_e
    s = np.sqrt(np.maximum(v, 0))
    ax.plot(xg, m, "r")
    ax.fill_between(xg, m - 2 * s, m + 2 * s, color="r", alpha=0.2)
    ax.scatter(X_tr, y_tr, color="k", s=10)
    ax.set_ylim(-4, 4)
    ax.set_title(f"ℓ = {ell}: log-ML = {lml_e:.1f}")
plt.show()
print("log marginal likelihood theo ℓ:", {k: round(v, 2) for k, v in lml_by_ell.items()})

ℓ quá nhỏ: hàm "uốn" qua mọi điểm, bất định lớn ngay giữa các điểm; ℓ quá lớn: gần như đường thẳng, không theo kịp dữ liệu. Log marginal likelihood cao nhất ở mức vừa phải. `GaussianProcessRegressor` tối đa nó bằng L-BFGS (nhiều điểm khởi đầu với `n_restarts_optimizer`), học cả mức nhiễu nhờ `WhiteKernel`:

In [ ]:
kernel = ConstantKernel(1.0) * RBF(1.0) + WhiteKernel(0.1)
gpr = GaussianProcessRegressor(kernel, n_restarts_optimizer=5, random_state=0).fit(X_tr[:, None], y_tr)
print("kernel học được:", gpr.kernel_)
print(f"log marginal likelihood = {gpr.log_marginal_likelihood_value_:.2f}")

## 5. Kernel quyết định ngoại suy

Chuỗi có **xu hướng tuyến tính + mùa vụ** (như doanh số). Train trên $x \in [0, 10]$, dự báo $x \in (10, 15]$:

In [ ]:
x5 = np.sort(rng.uniform(0, 10, 60))
f5 = lambda x: 0.5 * x + np.sin(2 * np.pi * x / 2.5)
y5 = f5(x5) + rng.normal(0, 0.15, 60)
xg5 = np.linspace(0, 15, 400)
k_rbf = ConstantKernel(1.0) * RBF(1.0) + WhiteKernel(0.1)
k_struct = ConstantKernel(0.1) * DotProduct(1.0) + ConstantKernel(1.0) * ExpSineSquared(1.0, 2.5) + WhiteKernel(0.1)
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, (name, k) in zip(axes, [("RBF", k_rbf), ("Linear + Periodic", k_struct)]):
    g = GaussianProcessRegressor(k, n_restarts_optimizer=3, random_state=0, normalize_y=True).fit(x5[:, None], y5)
    m, s = g.predict(xg5[:, None], return_std=True)
    future = xg5 > 10
    rmse = np.sqrt(np.mean((m[future] - f5(xg5[future])) ** 2))
    ax.plot(xg5, f5(xg5), "g--", lw=1)
    ax.plot(xg5, m, "r")
    ax.fill_between(xg5, m - 2 * s, m + 2 * s, color="r", alpha=0.2)
    ax.scatter(x5, y5, s=8, color="k")
    ax.axvline(10, color="gray", ls=":")
    ax.set_title(f"{name}: RMSE ngoại suy (10, 15] = {rmse:.2f} | log-ML = {g.log_marginal_likelihood_value_:.1f}")
    print(f"{name:18s} RMSE ngoại suy = {rmse:.2f} | log-ML = {g.log_marginal_likelihood_value_:.1f}")
plt.show()

Cùng dữ liệu, cùng thuật toán — khác **kernel** (tức khác giả định). RBF chỉ biết "hàm trơn" → ngoài dữ liệu quay về trung bình. Kernel có cấu trúc (tuyến tính + tuần hoàn) mã hoá đúng xu hướng và mùa vụ → ngoại suy tốt, và log marginal likelihood (trên dữ liệu train) cũng chọn đúng kernel này. Thiết kế kernel = feature engineering của GP.

## 6. Chi phí và giới hạn

Huấn luyện cần Cholesky của ma trận $n \times n$: thời gian $O(n^3)$, bộ nhớ $O(n^2)$.

In [ ]:
for n in [500, 1000, 2000, 4000]:
    Xn = rng.uniform(0, 10, n)
    K = rbf(Xn, Xn) + 0.01 * np.eye(n)
    t0 = time.perf_counter()
    np.linalg.cholesky(K)
    print(f"n = {n:4d}: Cholesky {time.perf_counter() - t0:.3f}s | ma trận kernel {K.nbytes / 1e6:.0f} MB")

Lý thuyết: gấp đôi $n$ → thời gian ×8, bộ nhớ ×4 (bộ nhớ đúng ×4; thời gian đo ở $n$ nhỏ tăng chậm hơn ×8 nhờ BLAS đa luồng và chi phí cố định, nhưng tiến dần về ×8 khi $n$ lớn). GP chính xác thực tế giới hạn ở vài chục nghìn điểm. Hướng mở rộng: **sparse GP / inducing points** (xấp xỉ bằng $m \ll n$ điểm đại diện, $O(nm^2)$), random Fourier features, thư viện GPU (GPyTorch).

| Ưu điểm | Nhược điểm |
|---|---|
| độ bất định có nguyên tắc, hiệu chỉnh tốt khi kernel hợp | $O(n^3)$ thời gian, $O(n^2)$ bộ nhớ |
| ít dữ liệu vẫn tốt; siêu tham số học từ log-ML | kernel chuẩn (RBF) kém khi số chiều lớn |
| kernel mã hoá kiến thức (tuần hoàn, xu hướng) | nhiễu không Gaussian/phân loại cần xấp xỉ |

**Ứng dụng tiêu biểu**: **Bayesian optimization** (chương 16 — GP làm mô hình thay thế, chọn điểm thử tiếp theo theo trung bình và độ bất định), mô phỏng đắt (kỹ thuật, khoa học), kriging trong địa thống kê, chuỗi thời gian ít điểm.

## 🧠 Kiểm tra nhanh

1. GP là gì? Liên hệ với hồi quy tuyến tính Bayes qua kernel thế nào?
2. Length scale ℓ ảnh hưởng gì tới hàm lấy mẫu và tới dự đoán?
3. Viết công thức trung bình và phương sai hậu nghiệm. Nó đến từ đâu?
4. Log marginal likelihood cân bằng hai yếu tố nào?
5. Vì sao GP với RBF không ngoại suy được xu hướng? Làm sao khắc phục?
6. Chi phí của GP chính xác? Cách mở rộng cho dữ liệu lớn?

## 📌 Tóm tắt

- GP = phân phối trên hàm, xác định bởi kernel; dự đoán = Gaussian có điều kiện; cài đặt bằng Cholesky.
- Siêu tham số (ℓ, σ_f, σ_n) học bằng tối đa log marginal likelihood; kernel quyết định ngoại suy.
- Độ bất định có nguyên tắc nhưng $O(n^3)$ — dùng cho dữ liệu nhỏ/vừa, Bayesian optimization, mô phỏng đắt.

## 📚 Tài liệu tham khảo

- Rasmussen & Williams (2006) — *Gaussian Processes for Machine Learning* (miễn phí: gaussianprocess.org/gpml), ch. 2 & 5
- Görtler, Kehlbeck & Deussen (2019) — *A Visual Exploration of Gaussian Processes* (Distill)
- Duvenaud (2014) — *The Kernel Cookbook*